In [65]:
import pandas as pd
import urllib.parse
import warnings
import sys
# Add utils folder to path
sys.path.append(r'C://Python_scripts')
warnings.filterwarnings("ignore")
import logging
from utils.log_file import LogManager
from utils.db_handler import *


# Set up logging
log_manager = LogManager("mkt_data_summary")
logger = logging.getLogger(__name__)
logger.info("Starting Marketing data processing script")

In [66]:
def get_google_sheets_data():
    """Fetch data from Google Sheets"""
    try:
        # Sheet info
        sheet_id = "1YYNZXrQiKfwgxzuDy1FfdeBGagxeB8djCYaaLHsk03A"
        sheet_name = "Overall (Adj)"

        # URL encode the sheet name
        encoded_sheet_name = urllib.parse.quote(sheet_name)

        # Build the URL
        url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={encoded_sheet_name}"

        # Read into DataFrame
        df = pd.read_csv(url)
        df.head(5)
        logger.info("Successfully retrieved %s records from Google Sheets", len(df))
        return df
    except Exception as e:
        logger.error("Error fetching Google Sheets data: %s", str(e))
        raise

In [67]:
df_raw = get_google_sheets_data()
df_raw.columns.to_list()

['Unnamed: 0',
 'Date ',
 'Day  ',
 'Campaign ',
 'Spend Taka',
 'Exchange Rate',
 'USD',
 'Spend Acquistion (Top Funnel)',
 'Retargeting (Mid & Bottom Funnel)',
 'Branding (Awareness)',
 'Traffic Data Impressions ',
 'CPM / 1,000 ',
 'Clicks ',
 'CPC ',
 'CTR ',
 'App Installs Overall',
 'Organic ',
 'Paid',
 'CPI Overall',
 'Paid.1',
 'CTI ',
 'DAU (Web+App)',
 'RTN DAU',
 'New DAU',
 'App',
 'App (Organic) ',
 'App (Paid)',
 'Unpaid',
 'CP DAU ',
 'Click / DAU ',
 'Drop-off ',
 'MAU ',
 'DAU/MAU ',
 'Sessions ',
 'CP Session ',
 'Add to Cart ',
 'DAU / A2C ',
 'Checkout ',
 'A2C / Checkout ',
 'GA4 Report Purchase',
 'New Purchase',
 'Rtn Purchase',
 '1 Day Rtn Purchase',
 '2 - 7 Days Rtn Purchase',
 '8 - 30 Days Rtn Purchase',
 '31 - 90 Days Purchase',
 'Business Data (CMS) Gross Buyers',
 'Net Buyers ',
 'CPB (Gross)',
 'CPB (Net)',
 'CR',
 'New Buyers',
 'CAC',
 'Returning Buyers',
 'CPRB',
 'New Buyer (%)',
 'Gross Orders',
 'Net Orders',
 'CPO (Gross)',
 'CPO (Net)',
 'Order / 

In [68]:
# Rename duplicate columns to avoid duplicacy
def rename_duplicate_columns(df):
    """Rename duplicate columns by appending a suffix"""
    cols = df.columns.tolist()
    col_counts = {}
    new_cols = []
    
    for col in cols:
        # Strip whitespace for duplicate detection
        col_stripped = col.strip()
        
        if col_stripped in col_counts:
            col_counts[col_stripped] += 1
            new_col = f"{col}_{col_counts[col_stripped]}"
        else:
            col_counts[col_stripped] = 0
            new_col = col
        
        new_cols.append(new_col)
    
    df.columns = new_cols
    print(f"Renamed columns to handle duplicates:\n{df.columns.tolist()}")
    return df

# Apply the renaming to df_raw
df_raw = rename_duplicate_columns(df_raw)
print(f"\nTotal columns: {len(df_raw.columns)}")


Renamed columns to handle duplicates:
['Unnamed: 0', 'Date ', 'Day  ', 'Campaign ', 'Spend Taka', 'Exchange Rate', 'USD', 'Spend Acquistion (Top Funnel)', 'Retargeting (Mid & Bottom Funnel)', 'Branding (Awareness)', 'Traffic Data Impressions ', 'CPM / 1,000 ', 'Clicks ', 'CPC ', 'CTR ', 'App Installs Overall', 'Organic ', 'Paid', 'CPI Overall', 'Paid.1', 'CTI ', 'DAU (Web+App)', 'RTN DAU', 'New DAU', 'App', 'App (Organic) ', 'App (Paid)', 'Unpaid', 'CP DAU ', 'Click / DAU ', 'Drop-off ', 'MAU ', 'DAU/MAU ', 'Sessions ', 'CP Session ', 'Add to Cart ', 'DAU / A2C ', 'Checkout ', 'A2C / Checkout ', 'GA4 Report Purchase', 'New Purchase', 'Rtn Purchase', '1 Day Rtn Purchase', '2 - 7 Days Rtn Purchase', '8 - 30 Days Rtn Purchase', '31 - 90 Days Purchase', 'Business Data (CMS) Gross Buyers', 'Net Buyers ', 'CPB (Gross)', 'CPB (Net)', 'CR', 'New Buyers', 'CAC', 'Returning Buyers', 'CPRB', 'New Buyer (%)', 'Gross Orders', 'Net Orders', 'CPO (Gross)', 'CPO (Net)', 'Order / Buyer', 'Revenue', 'AO

In [69]:
def process_data(df):
    """Process and transform the raw data"""
    try:
        logger.info("Starting data processing and transformation")

        # Convert all column names to lowercase and remove extra spaces
        df.columns = df.columns.str.lower().str.strip()
        df.columns = df.columns.str.replace(" ", "_")
        df.columns = df.columns.str.replace("/", "_")
        df.columns = df.columns.str.replace("&", "and")
        df.columns = df.columns.str.replace("(", "").str.replace(")", "")
        print(df.columns.to_list())
        
        # Remove duplicate unnamed columns
        df = df.drop(columns=[col for col in df.columns if col.startswith("unnamed")], errors="ignore")
        
        # Handle date column safely
        if "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"], errors="coerce")

        # Process currency columns - clean then convert to numeric
        currency_columns = [
            "spend_taka", "usd", "spend_acquistion_top_funnel",
            "retargeting_mid_and_bottom_funnel", "branding_awareness",
            "cpm___1,000", "cpc", "cpi_overall", "paid.1", "cp_dau","paid.2",
            "cp_session", "cpb_gross", "cpb_net", "cac", "cprb",
            "cpo_gross", "cpo_net", "revenue", "aov", "cpo_overall",
            "paid_2", "cp_fb_purchase", "cp_gg_purchase", "exchange_rate","revenue__1","aov__1"
        ]
        for col in currency_columns:
            if col in df.columns:
                try:
                    # Remove common currency symbols and commas, then convert
                    df[col] = (
                        df[col]
                        .astype(str)
                        .str.replace("$", "", regex=False)
                        .str.replace("৳", "", regex=False)
                        .str.replace(",", "", regex=False)
                        .str.strip()
                    )
                    df[col] = pd.to_numeric(df[col], errors="coerce")
                    logger.debug(f"Processed currency column '{col}': sample = {df[col].dropna().head(1).tolist()}")
                except Exception as e:
                    logger.warning(f"Error processing currency column '{col}': {str(e)}")

        # Process percentage columns - divide by 100 only if values are > 1
        # First, identify actual percentage columns dynamically
        percentage_indicators = ["ctr", "cr", "click__dau", "dau___a2c", "a2c___checkout", "checkout___purchase"]
        actual_percentage_cols = [col for col in df.columns if any(ind in col for ind in percentage_indicators)]
        
        # Also add explicit percentage columns
        percentage_columns = [
            "ctr", "cti", "cr", "cir", "new_buyer_","dau_mau", "click___dau","new_buyer_%",
            "organic_%", "paid_%", "google_%___paid", "facebook_%___paid",
            "crm_", "others_"
        ]
        
        # Combine and deduplicate
        all_percentage_cols = list(set(actual_percentage_cols + percentage_columns))
        print(all_percentage_cols)
        
        for col in all_percentage_cols:
            if col in df.columns:
                try:
                    # Remove percentage sign and commas, then convert
                    df[col] = (
                        df[col]
                        .astype(str)
                        .str.replace("%", "", regex=False)
                        .str.replace(",", "", regex=False)
                        .str.strip()
                    )
                    numeric_col = pd.to_numeric(df[col], errors="coerce")
                    
                    # Only divide by 100 if max value > 1 (indicating it's in percentage form)
                    if numeric_col.max() > 1:
                        df[col] = numeric_col / 100
                    else:
                        df[col] = numeric_col
                except Exception as e:
                    logger.warning(f"Error processing percentage column '{col}': {str(e)}")

        # Get all existing columns - maintain original order (not sorted)
        existing_columns = [col for col in df.columns if not col.startswith("unnamed")]
        logger.info(f"Processing {len(existing_columns)} columns")
        df = df[existing_columns]

        # Remove duplicate columns
        df = df.loc[:, ~df.columns.duplicated(keep="first")]

        # Rename columns to standardized format (using lowercase transformed column names)
        df = df.rename(
            columns={
                "date": "date",
                "day": "day",
                "campaign": "campaign",
                "spend_taka": "spend_taka",
                "usd": "spend_in_usd",
                "spend_acquistion_top_funnel": "spend_acquisition_top_funnel",
                "retargeting_mid_and_bottom_funnel": "spend_retargeting_mid_and_bottom_funnel",
                "branding_awareness": "spend_branding_awareness",
                "traffic_data_impressions": "impressions",
                "cpm___1,000": "cpm",
                "clicks": "clicks",
                "cpc": "cpc",
                "ctr": "ctr",
                "app_installs_overall": "app_installs_overall",
                "organic": "app_installs_organic",
                "paid": "app_installs_paid",
                "cpi_overall": "cpi_overall",
                "paid.1": "cpi_paid",
                "cti": "cti",
                "dau_web+app": "dau_web_and_app",
                "rtn_dau": "returning_dau",
                "new_dau": "new_dau",
                "app": "app_dau",
                "app_organic": "app_dau_organic",
                "app_paid": "app_dau_paid",
                "unpaid": "unpaid_dau",
                "cp_dau": "cost_per_dau",
                "click___dau": "click_per_dau",
                "drop-off": "drop_off_rate",
                "mau": "mau",
                "dau_mau": "dau_per_mau",
                "sessions": "sessions",
                "cp_session": "cost_per_session",
                "add_to_cart": "add_to_cart",
                "dau___a2c": "dau_per_add_to_cart",
                "checkout": "checkout",
                "a2c___checkout": "add_to_cart_per_checkout",
                "ga4_report_purchase": "ga4_reported_purchase",
                "new_purchase": "ga4_reported_new_purchase",
                "rtn_purchase": "ga4_reported_returning_purchase",
                "1_day_rtn_purchase": "ga4_reported_1_day_returning_purchase",
                "2_-_7_days_rtn_purchase": "ga4_reported_2_to_7_days_returning_purchase",
                "8_-_30_days_rtn_purchase": "ga4_reported_8_to_30_days_returning_purchase",
                "31_-_90_days_purchase": "ga4_reported_31_to_90_days_purchase",
                "business_data_cms_gross_buyers": "cms_reported_gross_buyers",
                "net_buyers": "cms_reported_net_buyers",
                "cpb_gross": "cms_reported_cost_per_buyer_gross",
                "cpb_net": "cms_reported_cost_per_buyer_net",
                "cr": "cms_reported_conversion_rate",
                "new_buyers": "cms_reported_new_buyers",
                "new_buyer_%": "cms_reported_new_buyer_percentage",
                "gross_orders": "cms_reported_gross_orders",
                "net_orders": "cms_reported_net_orders",
                "cpo_gross": "cms_reported_cost_per_order_gross",
                "cpo_net": "cms_reported_cost_per_order_net",
                "order___buyer": "cms_reported_order_per_buyer",
                "revenue": "cms_reported_gross_revenue",
                "aov": "cms_reported_average_order_value",
                "cir": "cms_reported_customer_investment_return",
                "adjust_data_purchase_orders_gross_orders": "adjust_gross_orders",
                "canceled_orders": "adjust_canceled_orders",
                "organic_%": "adjust_organic_order_percentage",
                "paid_%": "adjust_paid_order_percentage",
                "google_%___paid": "adjust_google_paid_order_percentage",
                "facebook_%___paid": "adjust_facebook_paid_order_percentage",
                "crm_%": "adjust_crm_order_percentage",
                "others_%": "adjust_others_order_percentage",
                "cpo_overall": "adjust_cpo_overall",
                "paid.2": "adjust_paid_cpo",
                "cp_fb_purchase": "adjust_cp_fb_purchase_cpo",
                "cp_gg_purchase": "adjust_cp_gg_purchase_cpo",
                "checkout___purchase": "adjust_checkout_per_purchase",
                "revenue__1": "adjust_revenue",
                "aov__1": "adjust_aov",
                "roas": "adjust_roas",
            }
        )

        logger.info("Successfully processed %s records with %s columns", len(df), len(df.columns))
        logger.info("Final columns: %s", df.columns.tolist())
        return df

    except Exception as e:
        logger.error("Error processing data: %s", str(e))
        import traceback
        traceback.print_exc()
        raise

In [70]:
df_processed = process_data(df_raw)
df_processed

['unnamed:_0', 'date', 'day', 'campaign', 'spend_taka', 'exchange_rate', 'usd', 'spend_acquistion_top_funnel', 'retargeting_mid_and_bottom_funnel', 'branding_awareness', 'traffic_data_impressions', 'cpm___1,000', 'clicks', 'cpc', 'ctr', 'app_installs_overall', 'organic', 'paid', 'cpi_overall', 'paid.1', 'cti', 'dau_web+app', 'rtn_dau', 'new_dau', 'app', 'app_organic', 'app_paid', 'unpaid', 'cp_dau', 'click___dau', 'drop-off', 'mau', 'dau_mau', 'sessions', 'cp_session', 'add_to_cart', 'dau___a2c', 'checkout', 'a2c___checkout', 'ga4_report_purchase', 'new_purchase', 'rtn_purchase', '1_day_rtn_purchase', '2_-_7_days_rtn_purchase', '8_-_30_days_rtn_purchase', '31_-_90_days_purchase', 'business_data_cms_gross_buyers', 'net_buyers', 'cpb_gross', 'cpb_net', 'cr', 'new_buyers', 'cac', 'returning_buyers', 'cprb', 'new_buyer_%', 'gross_orders', 'net_orders', 'cpo_gross', 'cpo_net', 'order___buyer', 'revenue', 'aov', 'cir', 'adjust_data_purchase_orders_gross_orders', 'canceled_orders', 'net_order

,date,day,campaign,spend_taka,exchange_rate,spend_in_usd,spend_acquisition_top_funnel,spend_retargeting_mid_and_bottom_funnel,spend_branding_awareness,impressions,...,adjust_crm_order_percentage,adjust_others_order_percentage,adjust_cpo_overall,adjust_paid_cpo,adjust_cp_fb_purchase_cpo,adjust_cp_gg_purchase_cpo,adjust_checkout_per_purchase,adjust_revenue,adjust_aov,adjust_roas
0,2024-10-24,Thursday,NaN,601.0,117.85,5.0,0.0,0.0,5.0,"10,363",...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0
1,2024-10-25,Friday,NaN,594.0,118.09,5.0,0.0,0.0,5.0,"15,433",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,2024-10-26,Saturday,NaN,554.0,118.09,5.0,0.0,0.0,5.0,"11,602",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,2024-10-27,Sunday,NaN,562.0,NaN,5.0,0.0,0.0,5.0,"10,392",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,2024-10-28,Monday,NaN,599.0,117.77,5.0,0.0,0.0,5.0,"8,705",...,NaN,NaN,120.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
847,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
848,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
849,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
850,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [71]:
df_processed.columns.to_list()

['date',
 'day',
 'campaign',
 'spend_taka',
 'exchange_rate',
 'spend_in_usd',
 'spend_acquisition_top_funnel',
 'spend_retargeting_mid_and_bottom_funnel',
 'spend_branding_awareness',
 'impressions',
 'cpm',
 'clicks',
 'cpc',
 'ctr',
 'app_installs_overall',
 'app_installs_organic',
 'app_installs_paid',
 'cpi_overall',
 'cpi_paid',
 'cti',
 'dau_web_and_app',
 'returning_dau',
 'new_dau',
 'app_dau',
 'app_dau_organic',
 'app_dau_paid',
 'unpaid_dau',
 'cost_per_dau',
 'click_per_dau',
 'drop_off_rate',
 'mau',
 'dau_per_mau',
 'sessions',
 'cost_per_session',
 'add_to_cart',
 'dau_per_add_to_cart',
 'checkout',
 'add_to_cart_per_checkout',
 'ga4_reported_purchase',
 'ga4_reported_new_purchase',
 'ga4_reported_returning_purchase',
 'ga4_reported_1_day_returning_purchase',
 'ga4_reported_2_to_7_days_returning_purchase',
 'ga4_reported_8_to_30_days_returning_purchase',
 'ga4_reported_31_to_90_days_purchase',
 'cms_reported_gross_buyers',
 'cms_reported_net_buyers',
 'cms_reported_cos

In [72]:
# Create PostgreSQL table and upload data
from sqlalchemy import create_engine, MetaData, Table, Column, String, Float, DateTime, Integer, inspect, text
from sqlalchemy.types import TEXT

def get_postgres_datatype(pandas_dtype):
    """Map pandas dtypes to PostgreSQL dtypes"""
    dtype_str = str(pandas_dtype)
    
    if 'datetime' in dtype_str:
        return 'TIMESTAMP'
    elif 'int' in dtype_str:
        return 'INTEGER'
    elif 'float' in dtype_str:
        return 'NUMERIC(15,4)'
    elif 'object' in dtype_str or 'string' in dtype_str:
        return 'VARCHAR(255)'
    else:
        return 'VARCHAR(255)'

def sanitize_column_name(col_name):
    """Quote column names that contain problematic special characters"""
    # Only quote if column has truly problematic characters (NOT underscores - those are fine)
    # Problematic: comma, slash, period, parentheses, dash, space, percent, ampersand
    problematic_chars = [',', '/', '.', '(', ')', '-', ' ', '%', '&']
    has_problematic = any(c in col_name for c in problematic_chars)
    starts_with_digit = col_name[0].isdigit() if col_name else False
    
    if has_problematic or starts_with_digit:
        return f'"{col_name}"'
    return col_name

def create_table_and_upload(df, table_name="daily_mkt_data_summary"):
    """
    Create PostgreSQL table based on DataFrame schema and upload data
    
    Args:
        df: DataFrame with processed data
        table_name: Name of the table to create
    """
    try:
        logger.info(f"Starting table creation and data upload for: {table_name}")

        # Get database credentials from DB_CONFIG_2
        db_host = DB_CONFIG_2['host']
        db_port = DB_CONFIG_2['port']
        db_name = DB_CONFIG_2['dbname']
        db_user = DB_CONFIG_2['user']
        db_password = DB_CONFIG_2['password']
        
        print(f"\nDatabase Connection Details:")
        print(f"  Host: {db_host}")
        print(f"  Port: {db_port}")
        print(f"  Database: {db_name}")
        print(f"  User: {db_user}\n")
        
        # Create connection string
        connection_string = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
        
        # Create SQLAlchemy engine
        engine = create_engine(connection_string, pool_pre_ping=True, pool_recycle=3600)
        
        # Test connection
        with engine.connect() as connection:
            connection.execute(text("SELECT 1"))
            logger.info("Successfully connected to PostgreSQL database")
            print("✓ Successfully connected to PostgreSQL\n")
        
        # Generate CREATE TABLE statement
        print("="*80)
        print(f"TABLE SCHEMA FOR: {table_name}")
        print("="*80)
        
        create_table_sql = f"CREATE TABLE IF NOT EXISTS {table_name} (\n"
        create_table_sql += "    id SERIAL PRIMARY KEY,\n"
        
        for col in df.columns:
            safe_col = sanitize_column_name(col)
            pg_dtype = get_postgres_datatype(df[col].dtype)
            create_table_sql += f"    {safe_col} {pg_dtype},\n"
        
        # Remove last comma and close statement
        create_table_sql = create_table_sql.rstrip(',\n') + "\n);"
        
        print(create_table_sql)
        print("="*80 + "\n")
        
        # Log the schema
        logger.info(f"Generated CREATE TABLE statement:\n{create_table_sql}")
        
        # Display DataFrame info
        print(f"\nDataFrame Info:")
        print(f"  Total Rows: {len(df)}")
        print(f"  Total Columns: {len(df.columns)}")
        print(f"\nColumn Data Types:")
        for col, dtype in df.dtypes.items():
            safe_col = sanitize_column_name(col)
            pg_dtype = get_postgres_datatype(dtype)
            print(f"  {col:40s} | pandas: {str(dtype):15s} | PostgreSQL: {pg_dtype}")
        
        # Create the table first
        with engine.connect() as connection:
            # Drop table if exists
            try:
                connection.execute(text(f"DROP TABLE IF EXISTS {table_name} CASCADE"))
                connection.commit()
                logger.info(f"Dropped existing table: {table_name}")
                print(f"\n✓ Dropped existing table: {table_name}")
            except Exception as e:
                logger.warning(f"Could not drop table: {str(e)}")
            
            # Execute CREATE TABLE statement with quoted column names
            create_stmt = (f"CREATE TABLE IF NOT EXISTS {table_name} (\n" +
                          "    id SERIAL PRIMARY KEY,\n" +
                          ",\n".join([f"    {sanitize_column_name(col)} {get_postgres_datatype(df[col].dtype)}" 
                                     for col in df.columns]) +
                          "\n)")
            connection.execute(text(create_stmt))
            connection.commit()
            logger.info(f"Successfully created table: {table_name}")
            print(f"✓ Successfully created table: {table_name}")
        
        # Upload data to PostgreSQL
        print(f"\nUploading {len(df)} records to PostgreSQL...\n")
        
        # Rename columns to use quoted names for upload
        df_upload = df.copy()
        df_upload.columns = [sanitize_column_name(col) for col in df_upload.columns]
        
        df_upload.to_sql(
            table_name,
            engine,
            if_exists='append',  # Append to newly created table
            index=False,
            chunksize=500,
            method='multi'
        )
        
        logger.info(f"Successfully uploaded {len(df)} records to {table_name}")
        print(f"✓ Successfully uploaded {len(df)} records to PostgreSQL")
        
        # Verify upload
        with engine.connect() as connection:
            result = connection.execute(text(f"SELECT COUNT(*) FROM {table_name}"))
            row_count = result.fetchone()[0]
            logger.info(f"Verification: {row_count} rows in {table_name} table")
            print(f"✓ Verification: {row_count} rows successfully uploaded to {table_name}")
        
        return True
        
    except Exception as e:
        logger.error(f"Error creating table or uploading data: {str(e)}")
        import traceback
        traceback.print_exc()
        return False

# Execute table creation and data upload
print("Starting PostgreSQL table creation and data upload process...\n")
if create_table_and_upload(df_processed):
    print("\n✓ Table created and data successfully uploaded to PostgreSQL")
else:
    print("\n✗ Failed to create table or upload data to PostgreSQL")

Starting PostgreSQL table creation and data upload process...


Database Connection Details:
  Host: 172.16.90.18
  Port: 5432
  Database: postgres
  User: postgres

✓ Successfully connected to PostgreSQL

TABLE SCHEMA FOR: daily_mkt_data_summary
CREATE TABLE IF NOT EXISTS daily_mkt_data_summary (
    id SERIAL PRIMARY KEY,
    date TIMESTAMP,
    day VARCHAR(255),
    campaign VARCHAR(255),
    spend_taka NUMERIC(15,4),
    exchange_rate NUMERIC(15,4),
    spend_in_usd NUMERIC(15,4),
    spend_acquisition_top_funnel NUMERIC(15,4),
    spend_retargeting_mid_and_bottom_funnel NUMERIC(15,4),
    spend_branding_awareness NUMERIC(15,4),
    impressions VARCHAR(255),
    cpm NUMERIC(15,4),
    clicks VARCHAR(255),
    cpc NUMERIC(15,4),
    ctr NUMERIC(15,4),
    app_installs_overall VARCHAR(255),
    app_installs_organic VARCHAR(255),
    app_installs_paid VARCHAR(255),
    cpi_overall NUMERIC(15,4),
    cpi_paid NUMERIC(15,4),
    cti NUMERIC(15,4),
    dau_web_and_app VARCHAR(255),
    r